In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

project_path = r"C:\Users\Lenovo\Data_Analysis_Business\Project15_pricing_promotion_dashboard"
data_path = project_path + r"\data"

print("Setup complete")

Setup complete


### Create product catalogue

In [2]:
categories = {
    "Electronics": [
        "Wireless Charger", "USB Hub", "Bluetooth Speaker", "Laptop Stand", "Webcam",
        "Phone Tripod", "Portable Power Bank", "LED Desk Lamp"
    ],
    "Home & Kitchen": [
        "Air Fryer Liners", "Storage Basket", "Kitchen Scale", "Coffee Grinder",
        "Food Container Set", "Dish Rack", "Reusable Water Bottle", "Spice Rack"
    ],
    "Fitness": [
        "Yoga Mat", "Foam Roller", "Resistance Bands", "Massage Ball",
        "Ankle Weights", "Gym Gloves", "Protein Shaker", "Skipping Rope"
    ],
    "Beauty": [
        "Face Serum", "LED Mirror", "Makeup Organiser", "Scalp Massager",
        "Hair Towel", "Nail Kit", "Travel Toiletry Bag", "Facial Roller"
    ],
    "Pet Supplies": [
        "Dog Toy", "Cat Brush", "Pet Blanket", "Treat Pouch",
        "Training Pads", "Collapsible Bowl", "Pet Grooming Glove", "Litter Mat"
    ]
}

tiers = ["Budget", "Mid", "Premium"]

products = []

product_id = 1

for category, product_names in categories.items():
    for name in product_names:
        for tier in tiers:
            base_price = {
                "Budget": np.random.uniform(8, 18),
                "Mid": np.random.uniform(18, 40),
                "Premium": np.random.uniform(40, 90)
            }[tier]
            
            margin_pct = {
                "Budget": np.random.uniform(0.18, 0.35),
                "Mid": np.random.uniform(0.28, 0.48),
                "Premium": np.random.uniform(0.35, 0.60)
            }[tier]
            
            unit_cost = base_price * (1 - margin_pct)
            
            products.append({
                "product_id": f"SKU{product_id:03d}",
                "product_name": f"{name} {tier}",
                "category": category,
                "tier": tier,
                "standard_price": round(base_price, 2),
                "unit_cost": round(unit_cost, 2),
                "base_margin_pct": round(margin_pct, 3)
            })
            
            product_id += 1

products_df = pd.DataFrame(products)

products_df.head()

,product_id,product_name,category,tier,standard_price,unit_cost,base_margin_pct
0,SKU001,Wireless Charger Budget,Electronics,Budget,11.75,8.44,0.282
1,SKU002,Wireless Charger Mid,Electronics,Mid,37.06,26.53,0.284
2,SKU003,Wireless Charger Premium,Electronics,Premium,49.09,25.47,0.481
3,SKU004,USB Hub Budget,Electronics,Budget,12.32,9.81,0.204
4,SKU005,USB Hub Mid,Electronics,Mid,35.27,21.22,0.398


check number of products ran successfully

In [3]:
products_df.shape

(120, 7)

### Create customer base

In [4]:
num_customers = 5000

customer_segments = ["New", "Repeat", "VIP"]

segment_probs = [0.50, 0.40, 0.10]

regions = [
    "London",
    "South East",
    "South West",
    "Midlands",
    "North West",
    "Scotland",
    "Wales"
]

customers = []

for i in range(1, num_customers + 1):
    
    segment = np.random.choice(customer_segments, p=segment_probs)
    
    region = np.random.choice(regions)
    
    if segment == "New":
        avg_orders = np.random.randint(1, 3)
        
    elif segment == "Repeat":
        avg_orders = np.random.randint(3, 8)
        
    else:
        avg_orders = np.random.randint(8, 20)
    
    customers.append({
        "customer_id": f"CUST{i:05d}",
        "customer_segment": segment,
        "region": region,
        "expected_orders": avg_orders
    })

customers_df = pd.DataFrame(customers)

customers_df.head()

,customer_id,customer_segment,region,expected_orders
0,CUST00001,VIP,Midlands,10
1,CUST00002,New,Scotland,2
2,CUST00003,Repeat,South West,3
3,CUST00004,Repeat,Midlands,5
4,CUST00005,New,North West,1


In [5]:
customers_df["customer_segment"].value_counts()

customer_segment
New       2471
Repeat    2060
VIP        469
Name: count, dtype: int64

### Generate orders

In [6]:
marketing_channels = [
    "Google Ads",
    "Facebook Ads",
    "Email",
    "Organic",
    "TikTok",
    "Affiliate"
]

orders = []

order_id = 1

start_date = datetime(2024, 1, 1)

for _, customer in customers_df.iterrows():
    
    num_orders = customer["expected_orders"]
    
    for _ in range(num_orders):
        
        order_date = start_date + timedelta(
            days=np.random.randint(0, 365)
        )
        
        num_items = np.random.randint(1, 5)
        
        chosen_products = products_df.sample(num_items)
        
        marketing_channel = np.random.choice(
            marketing_channels,
            p=[0.25, 0.20, 0.15, 0.20, 0.10, 0.10]
        )
        
        for _, product in chosen_products.iterrows():
            
            units = np.random.randint(1, 4)
            
            # Discount behaviour
            if customer["customer_segment"] == "VIP":
                discount_pct = np.random.choice(
                    [0, 5, 10, 15, 20],
                    p=[0.15, 0.20, 0.30, 0.25, 0.10]
                )
                
            else:
                discount_pct = np.random.choice(
                    [0, 5, 10, 15],
                    p=[0.30, 0.35, 0.25, 0.10]
                )
            
            standard_price = product["standard_price"]
            
            final_price = standard_price * (1 - discount_pct / 100)
            
            revenue = final_price * units
            
            cost = product["unit_cost"] * units
            
            profit = revenue - cost
            
            margin_pct = (profit / revenue) if revenue > 0 else 0
            
            # Return logic
            returned = np.random.choice(
                [0, 1],
                p=[0.93, 0.07]
            )
            
            rating = np.random.choice(
                [3, 4, 5],
                p=[0.15, 0.35, 0.50]
            )
            
            orders.append({
                "order_id": f"ORD{order_id:06d}",
                "order_date": order_date,
                "customer_id": customer["customer_id"],
                "customer_segment": customer["customer_segment"],
                "region": customer["region"],
                "marketing_channel": marketing_channel,
                "product_id": product["product_id"],
                "product_name": product["product_name"],
                "category": product["category"],
                "tier": product["tier"],
                "units_sold": units,
                "standard_price": round(standard_price, 2),
                "discount_pct": discount_pct,
                "final_price": round(final_price, 2),
                "revenue": round(revenue, 2),
                "cost": round(cost, 2),
                "profit": round(profit, 2),
                "profit_margin_pct": round(margin_pct, 3),
                "returned": returned,
                "customer_rating": rating
            })
        
        order_id += 1

orders_df = pd.DataFrame(orders)

orders_df.head()

,order_id,order_date,customer_id,customer_segment,region,marketing_channel,product_id,product_name,category,tier,units_sold,standard_price,discount_pct,final_price,revenue,cost,profit,profit_margin_pct,returned,customer_rating
0,ORD000001,2024-08-24,CUST00001,VIP,Midlands,TikTok,SKU104,Pet Blanket Mid,Pet Supplies,Mid,1,34.46,15,29.29,29.29,22.26,7.03,0.240,0,3
1,ORD000001,2024-08-24,CUST00001,VIP,Midlands,TikTok,SKU049,Yoga Mat Budget,Fitness,Budget,2,10.87,10,9.78,19.57,17.68,1.89,0.096,0,5
2,ORD000001,2024-08-24,CUST00001,VIP,Midlands,TikTok,SKU012,Laptop Stand Premium,Electronics,Premium,1,43.73,15,37.17,37.17,26.25,10.92,0.294,0,4
3,ORD000002,2024-06-09,CUST00001,VIP,Midlands,Facebook Ads,SKU002,Wireless Charger Mid,Electronics,Mid,1,37.06,5,35.21,35.21,26.53,8.68,0.246,0,4
4,ORD000002,2024-06-09,CUST00001,VIP,Midlands,Facebook Ads,SKU075,Face Serum Premium,Beauty,Premium,1,89.30,0,89.30,89.30,56.18,33.12,0.371,0,5


In [7]:
orders_df.shape

(50977, 20)

### Add month & discount brands

In [8]:
orders_df["month"] = orders_df["order_date"].dt.strftime("%B")

orders_df["month_num"] = orders_df["order_date"].dt.month

orders_df["discount_band"] = pd.cut(
    orders_df["discount_pct"],
    bins=[-1, 0, 5, 10, 15, 100],
    labels=[
        "No Discount",
        "1-5%",
        "6-10%",
        "11-15%",
        "15%+"
    ]
)

orders_df.head()

,order_id,order_date,customer_id,customer_segment,region,marketing_channel,product_id,product_name,category,tier,...,final_price,revenue,cost,profit,profit_margin_pct,returned,customer_rating,month,month_num,discount_band
0,ORD000001,2024-08-24,CUST00001,VIP,Midlands,TikTok,SKU104,Pet Blanket Mid,Pet Supplies,Mid,...,29.29,29.29,22.26,7.03,0.240,0,3,August,8,11-15%
1,ORD000001,2024-08-24,CUST00001,VIP,Midlands,TikTok,SKU049,Yoga Mat Budget,Fitness,Budget,...,9.78,19.57,17.68,1.89,0.096,0,5,August,8,6-10%
2,ORD000001,2024-08-24,CUST00001,VIP,Midlands,TikTok,SKU012,Laptop Stand Premium,Electronics,Premium,...,37.17,37.17,26.25,10.92,0.294,0,4,August,8,11-15%
3,ORD000002,2024-06-09,CUST00001,VIP,Midlands,Facebook Ads,SKU002,Wireless Charger Mid,Electronics,Mid,...,35.21,35.21,26.53,8.68,0.246,0,4,June,6,1-5%
4,ORD000002,2024-06-09,CUST00001,VIP,Midlands,Facebook Ads,SKU075,Face Serum Premium,Beauty,Premium,...,89.30,89.30,56.18,33.12,0.371,0,5,June,6,No Discount


### Sort month order properly

In [9]:
month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

orders_df["month"] = pd.Categorical(
    orders_df["month"],
    categories=month_order,
    ordered=True
)

### Quality checks

In [10]:
orders_df.describe()

,order_date,units_sold,standard_price,discount_pct,final_price,revenue,cost,profit,profit_margin_pct,returned,customer_rating,month_num
count,50977,50977.000000,50977.000000,50977.000000,50977.000000,50977.000000,50977.000000,50977.000000,50977.000000,50977.000000,50977.000000,50977.000000
mean,2024-07-02 00:48:31.807285504,2.004218,36.933564,6.983149,34.364478,68.722326,42.731935,25.990412,0.320502,0.069188,4.351923,6.533123
min,2024-01-01 00:00:00,1.000000,8.060000,0.000000,6.450000,6.450000,5.610000,-0.480000,-0.020000,0.000000,3.000000,1.000000
25%,2024-04-03 00:00:00,1.000000,16.140000,0.000000,14.910000,27.990000,20.190000,6.370000,0.229000,0.000000,4.000000,4.000000
50%,2024-07-03 00:00:00,2.000000,30.980000,5.000000,28.060000,49.530000,33.990000,15.010000,0.305000,0.000000,5.000000,7.000000
75%,2024-09-30 00:00:00,3.000000,55.970000,10.000000,50.500000,89.360000,56.120000,34.950000,0.409000,0.000000,5.000000,9.000000
max,2024-12-30 00:00:00,3.000000,89.360000,20.000000,89.360000,268.080000,168.540000,152.280000,0.594000,1.000000,5.000000,12.000000
std,NaN,0.816305,24.470077,5.519875,22.917789,56.735450,31.431927,27.636886,0.121627,0.253776,0.726976,3.417810


In [11]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50977 entries, 0 to 50976
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           50977 non-null  object        
 1   order_date         50977 non-null  datetime64[ns]
 2   customer_id        50977 non-null  object        
 3   customer_segment   50977 non-null  object        
 4   region             50977 non-null  object        
 5   marketing_channel  50977 non-null  object        
 6   product_id         50977 non-null  object        
 7   product_name       50977 non-null  object        
 8   category           50977 non-null  object        
 9   tier               50977 non-null  object        
 10  units_sold         50977 non-null  int64         
 11  standard_price     50977 non-null  float64       
 12  discount_pct       50977 non-null  int64         
 13  final_price        50977 non-null  float64       
 14  revenu

### Create return adjusted profit

In [12]:
orders_df["adjusted_profit"] = np.where(
    orders_df["returned"] == 1,
    orders_df["profit"] * -0.5,
    orders_df["profit"]
)

orders_df["adjusted_profit"] = orders_df["adjusted_profit"].round(2)

### Introduce seasonality profit wise

In [13]:
# Fix: recreate unit_cost from existing cost before applying updated logic
orders_df["unit_cost"] = orders_df["cost"] / orders_df["units_sold"]

# Add stronger realism to the dataset
orders_df = orders_df.copy()

seasonality_map = {
    1: 0.82,
    2: 0.86,
    3: 0.92,
    4: 0.96,
    5: 1.00,
    6: 1.03,
    7: 1.00,
    8: 0.98,
    9: 1.06,
    10: 1.12,
    11: 1.28,
    12: 1.22
}

orders_df["seasonality_multiplier"] = orders_df["month_num"].map(seasonality_map)

channel_discount_boost = {
    "Google Ads": 1.05,
    "Facebook Ads": 1.08,
    "Email": 0.92,
    "Organic": 0.88,
    "TikTok": 1.15,
    "Affiliate": 1.02
}

orders_df["channel_discount_multiplier"] = orders_df["marketing_channel"].map(channel_discount_boost)

category_cost_multiplier = {
    "Electronics": 1.08,
    "Home & Kitchen": 1.00,
    "Fitness": 0.95,
    "Beauty": 0.90,
    "Pet Supplies": 0.98
}

orders_df["category_cost_multiplier"] = orders_df["category"].map(category_cost_multiplier)

orders_df["discount_effect"] = 1 - (
    (orders_df["discount_pct"] / 100) * orders_df["channel_discount_multiplier"]
)

orders_df["discount_effect"] = orders_df["discount_effect"].clip(lower=0.65)

orders_df["final_price"] = (
    orders_df["standard_price"]
    * orders_df["discount_effect"]
    * orders_df["seasonality_multiplier"]
)

orders_df["revenue"] = orders_df["final_price"] * orders_df["units_sold"]

orders_df["cost"] = (
    orders_df["unit_cost"]
    * orders_df["category_cost_multiplier"]
    * orders_df["units_sold"]
)

orders_df["profit"] = orders_df["revenue"] - orders_df["cost"]

orders_df["profit_margin_pct"] = orders_df["profit"] / orders_df["revenue"]

return_risk = {
    "Google Ads": 0.07,
    "Facebook Ads": 0.08,
    "Email": 0.04,
    "Organic": 0.05,
    "TikTok": 0.11,
    "Affiliate": 0.07
}

orders_df["return_probability"] = orders_df["marketing_channel"].map(return_risk)

orders_df["returned"] = np.random.binomial(
    1,
    orders_df["return_probability"]
)

orders_df["adjusted_profit"] = np.where(
    orders_df["returned"] == 1,
    orders_df["profit"] * -0.5,
    orders_df["profit"]
)

rating_probs = {
    "Google Ads": [0.16, 0.36, 0.48],
    "Facebook Ads": [0.18, 0.37, 0.45],
    "Email": [0.08, 0.30, 0.62],
    "Organic": [0.10, 0.32, 0.58],
    "TikTok": [0.24, 0.42, 0.34],
    "Affiliate": [0.14, 0.36, 0.50]
}

def generate_rating(channel):
    return np.random.choice([3, 4, 5], p=rating_probs[channel])

orders_df["customer_rating"] = orders_df["marketing_channel"].apply(generate_rating)

round_cols = [
    "unit_cost",
    "final_price",
    "revenue",
    "cost",
    "profit",
    "profit_margin_pct",
    "adjusted_profit"
]

for col in round_cols:
    orders_df[col] = orders_df[col].round(2)

orders_df.head()

,order_id,order_date,customer_id,customer_segment,region,marketing_channel,product_id,product_name,category,tier,...,month,month_num,discount_band,adjusted_profit,unit_cost,seasonality_multiplier,channel_discount_multiplier,category_cost_multiplier,discount_effect,return_probability
0,ORD000001,2024-08-24,CUST00001,VIP,Midlands,TikTok,SKU104,Pet Blanket Mid,Pet Supplies,Mid,...,August,8,11-15%,-3.07,22.26,0.98,1.15,0.98,0.8275,0.11
1,ORD000001,2024-08-24,CUST00001,VIP,Midlands,TikTok,SKU049,Yoga Mat Budget,Fitness,Budget,...,August,8,6-10%,2.06,8.84,0.98,1.15,0.95,0.8850,0.11
2,ORD000001,2024-08-24,CUST00001,VIP,Midlands,TikTok,SKU012,Laptop Stand Premium,Electronics,Premium,...,August,8,11-15%,7.11,26.25,0.98,1.15,1.08,0.8275,0.11
3,ORD000002,2024-06-09,CUST00001,VIP,Midlands,Facebook Ads,SKU002,Wireless Charger Mid,Electronics,Mid,...,June,6,1-5%,-3.73,26.53,1.03,1.08,1.08,0.9460,0.08
4,ORD000002,2024-06-09,CUST00001,VIP,Midlands,Facebook Ads,SKU075,Face Serum Premium,Beauty,Premium,...,June,6,No Discount,41.42,56.18,1.03,1.08,0.90,1.0000,0.08


check it worked

In [14]:
monthly_check = orders_df.groupby("month_num")[["revenue", "adjusted_profit"]].sum()
monthly_check["profit_margin_pct"] = monthly_check["adjusted_profit"] / monthly_check["revenue"]
monthly_check

,revenue,adjusted_profit,profit_margin_pct
month_num,,,
1,238376.80,55791.38,0.234047
2,231302.83,59092.82,0.255478
3,275036.37,83858.49,0.304900
4,274594.53,90130.74,0.328232
5,299362.44,106022.55,0.354161
6,286129.90,104345.00,0.364677
7,302369.55,105909.88,0.350266
8,299705.25,101664.43,0.339215
9,305836.87,114194.08,0.373382


### Export to CSV

In [15]:
orders_df.to_csv(
    data_path + r"\pricing_promotion_dataset.csv",
    index=False
)

print("CSV exported successfully")

CSV exported successfully


Export more realistic data set

In [18]:
orders_df.to_csv(
    data_path + r"\ecommerce_orders_realistic.csv",
    index=False
)

print("Realistic dataset exported successfully")

Realistic dataset exported successfully
